# Data sources

The main data source is the publication by [Sayaman et al.](https://doi.org/10.1016/j.immuni.2021.01.011).

The key datasets can be found here:  https://gdc.cancer.gov/about-data/publications/CCG-AIM-2020 (HRC Imputed Genotyping Data). Download all `chr_*.zip` and store their contents in `data/` directory (`*.dose.vcf.gz` files go into `data/imputed` while `*.info.gz` go into `data/info`). Note that these datasets are controlled and that you will need to obtain the appropriate permission via dbGaP. You might also find scripts from https://github.com/rwsayaman/TCGA_PanCancer_Immune_Genetics useful.

Other files can be fetched as follows:


In [ ]:
%%bash
mkdir -p data out

# RNA-seq expression data
curl -L https://api.gdc.cancer.gov/data/3586c0da-64d0-4b74-a449-5ff4d9136611 > data/EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.tsv
# TCGA IDs used in Sayaman et al.
curl -L https://api.gdc.cancer.gov/data/9df36037-3f9a-47aa-bcab-1c5554bb0443 | awk 'OFS="\t" {print $1"_"$1,$2"_"$2,$3,$4}' | sort > data/tcga-ids.txt
# UCSF PCA Ancestry: European ancestry (from Sayaman et al.)
curl -L https://api.gdc.cancer.gov/data/fdfa536a-c3c8-405d-99d9-bc9375b5084c | cut -d, -f2,6 | tr -d '"' | awk -F, '$2=="EUR" {print $1,$1}' > data/ucsf-eur.txt
# MAGMA's gene location file (hg19)
curl -L https://raw.githubusercontent.com/endeneon/MAGMA_analysis_protocol/refs/heads/main/data_files/Rev.NCBI37.3.gene.loc | awk '{print $6,$2,$3,$4,$5,$1}' OFS="\t" > data/NCBI37.3.gene.loc
# dbSNP v155 (hg19)
curl -L https://ftp.ncbi.nih.gov/snp/redesign/archive/b155/VCF/GCF_000001405.25.gz | gzip -c -d | awk '{print $1"_"$2,$3}' | sed -e 's/NC_000001.10/1/;s/NC_000002.11/2/;s/NC_000003.11/3/;s/NC_000004.11/4/;s/NC_000005.9/5/;s/NC_000006.11/6/;s/NC_000007.13/7/;s/NC_000008.10/8/;s/NC_000009.11/9/;s/NC_000010.10/10/;s/NC_000011.9/11/;s/NC_000012.11/12/;s/NC_000013.10/13/;s/NC_000014.8/14/;s/NC_000015.9/15/;s/NC_000016.9/16/;s/NC_000017.10/17/;s/NC_000018.9/18/;s/NC_000019.9/19/;s/NC_000020.10/20/;s/NC_000021.8/21/;s/NC_000022.10/22/;s/NW_004775435.1/25/' > data/dbsnp-v155-hg19.txt

Now, let's query the GDC database to get the project ID, sex and the age of each TCGA sample used in this analysis.
Ages are normalized and reported as z-scores.

In [2]:
import requests, json
import pandas as pd

# Get the list of TCGA IDs.
with open("data/tcga-ids.txt") as f:
    samples = list({l.split()[2] for l in f})

# Fetch all metadata from GDC in batches via GDC REST API.
hits = []
step = 300  # how many records to fetch per request; currently, API supports up to ~300 per request
for start in range(0, len(samples) * 10, step):
    d = json.loads(
        requests.get("https://api.gdc.cancer.gov/cases", params={
            "filters": json.dumps({
                "op": "in",
                "content": {
                    "field": "cases.submitter_id",
                    "value": samples[start:start + step]
                }
            }),
            "fields": ",".join([
                "submitter_id",
                "project.project_id",
                "demographic.gender",
                "demographic.age_at_index",
                "demographic.age_is_obfuscated",
            ]),
            "format": "JSON",
            "size": f"{step}",
        }).content.decode())
    hits += d["data"]["hits"]
    assert d["data"]["pagination"]["pages"] == 1
    start += step
    if start >= len(samples): break
print(f"fetching done; hits={len(hits)}")

# Form the dataframe and apply z-score normalization. Remove samples with no reported age.
df = pd.DataFrame(
    sorted([
        (d["submitter_id"], d["project"]["project_id"],
            d.get("demographic", {}).get("gender", 'X')[0].upper(),
            d.get("demographic", {}).get("age_at_index", None))
        for d in hits
    ]),
    columns=["sample", "type", "sex", "age"]
)
df.sex = df.sex.apply(lambda x: {'M': 1, 'F': 2}.get(x, 0))
df = df[~df.age.isna()]
df.age = (df.age - df.age.mean()) / df.age.std()
df[['sample', 'age', 'type', 'sex']].to_csv("out/tcga-covariates.csv", index=False, header=False, sep='\t')

fetching done; hits=10128


Also, let's prepare the phenotype and pathway data used for GWAS and GSA later on:

In [ ]:
%%bash
Rscript phenotypes.r data/EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.tsv data/pathways.txt
    # Takes ~30min

# Initialization

Here, we assume that the following tools are installed: 
- plink v2 (`plink2`)
- bcftools

In [1]:
import os
os.environ['R_HOME'] = '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/r/4.5.0/lib64/R'
import rpy2, pandas as pd

In [2]:
%load_ext rpy2.ipython

Let's install the R prerequisites as well:

In [ ]:
%%R
if (!require("BiocManager", quietly=TRUE))
    install.packages("BiocManager")
BiocManager::install(c(
    "GSVA", "GSEABase", "GSVAdata", "zFPKM", "biomaRt", "org.Hs.eg.db",
    "TCGAbiolinks", "TCGAretriever", "GWASinspector"), update=F, ask=F)
library(dplyr)
library(GWASinspector)
library(ggplot2)
library(GSVA)
library(GSEABase)
library(GSVAdata)
library(zFPKM)
library(stringr)
library(biomaRt)
library(data.table)
library(org.Hs.eg.db) # genome annotations for H.sapiens

# Data preparation

Here we filter imputed TCGA VCFs (with $R^2 \geq 0.5$ filter) and convert them to plink format.

In [ ]:
%%bash

# R2 filter
mkdir -p out/vcf
parallel --plus 'bcftools view -i "R2>=0.5" {} | bgzip -c > out/vcf/{/...}.vcf.gz' ::: data/imputed/*.dose.vcf.gz
    # Takes ~2h 30min
parallel 'tabix -p vcf {}' ::: out/vcf/*.vcf.gz
    # Takes ~20min

# Convert to plink format
mkdir -p out/tmp
parallel --plus 'plink2 --double-id --vcf {} --make-bed --out out/tmp/{/..}' ::: out/vcf/*.vcf.gz
    # Takes ~15min
# SNP id for the PLINK bim file
parallel "awk 'OFS=\"\t\" {ref=\$5; alt=\$6; if (ref>alt) { ref=\$6; alt=\$5 } print \$1,\$1\":\"\$4\":\"ref\":\"alt,\$3,\$4,\$5,\$6}' {} > out/tmp/{/}.updated" ::: out/tmp/chr*.bim
# Update IIDs and FIDs in the PLINK fam file from Birdseed file IDs to TCGA patient IDs
parallel 'plink2 --bfile {.} --bim {}.updated --update-ids data/tcga-ids.txt --make-pgen --out out/vcf/{/.}' ::: out/tmp/*.bim
# Merge all chromosomes in plink
plink2 --pmerge-list <(seq -f "out/vcf/chr%g" 1 22) --out out/vcf/hg19-merged
    # Takes ~5min

# Annotate SNPs
plink2 --pfile out/vcf/hg19-merged --make-bed --out out/vcf/hg19-merged
awk '{print $1"_"$4, $2}' out/vcf/hg19.bim |\
    awk 'FNR==NR {a[$1]=$2; next} ($1 in a) {print $1, $2, a[$1]}' data/dbsnp-v155-hg19.txt - |\
    awk '{print $2, $2":"$3}' > out/vcf/hg19-merged.bim.annotated
plink2 --pfile out/vcf/hg19-merged --update-name out/vcf/hg19-merged.bim.annotated --make-pgen --out out/vcf/hg19
    # Takes ~30min

# Data cleanup

Closely related individuals in the target data may lead to overfitted results, limiting the generalisability of the results.
Individuals that have a first or second degree relative in the sample ($π > 0.125$) can be removed as follows:

In [ ]:
%%bash

# Keep only EUR population
mkdir -p out/gwas-eur
plink2 --pfile out/vcf/hg19 --keep data/ucsf-eur.txt --out out/gwas-eur/hg19 --make-pgen
    # Takes ~1m

# Apply quality control to the subset
plink2 --pfile out/gwas-eur/hg19 --write-snplist --out out/gwas-eur/hg19 --maf 0.005 --geno 0.01 --mind 0.01
    # 1m
# 2. LD pruning to remove highly correlated SNPs
plink2 --pfile out/gwas-eur/hg19 --extract out/gwas-eur/hg19.snplist --out out/gwas-eur/hg19 --indep-pairwise 200 50 0.25
    # 1m
# Generate heterozygosity rates (HET) file
plink2 --pfile out/gwas-eur/hg19 --extract out/gwas-eur/hg19.prune.in --het --out out/gwas-eur/hg19
plink2 --pfile out/gwas-eur/hg19 --extract out/gwas-eur/hg19.snplist  --het --out out/gwas-eur/hg19-nold
Rscript correct_het.r out/gwas-eur/hg19.het out/gwas-eur/hg19.valid
Rscript correct_het.r out/gwas-eur/hg19-nold.het out/gwas-eur/hg19-nold.valid
    # 20s

# Calculate relatedness
plink2 --pfile out/gwas-eur/hg19 --extract out/gwas-eur/hg19.prune.in --keep out/gwas-eur/hg19.valid --king-cutoff 0.125 --out out/gwas-eur/hg19
plink2 --pfile out/gwas-eur/hg19 --keep out/gwas-eur/hg19-nold.valid --king-cutoff 0.125 --out out/gwas-eur/hg19-nold
    # Takes ~1 min

# Final QC
plink2 --pfile out/gwas-eur/hg19 --make-pgen --keep out/gwas-eur/hg19.king.cutoff.in.id --out out/gwas-eur/hg19-qc --extract out/gwas-eur/hg19.snplist
plink2 --pfile out/gwas-eur/hg19 --make-pgen --keep out/gwas-eur/hg19-nold.king.cutoff.in.id --out out/gwas-eur/hg19-nold-qc --extract out/gwas-eur/hg19.snplist
    # Takes ~2 min

# PCA
plink2 --pfile out/gwas-eur/hg19-qc --pca 10 --out out/gwas-eur/hg19-qc
plink2 --pfile out/gwas-eur/hg19-nold-qc --pca 10 --out out/gwas-eur/hg19-nold-qc
# -f1-11 selects IDs and top 9 PCs (to account for at least 90% of the variation)
paste <(cut -f1-11 out/gwas-eur/hg19-qc.eigenvec | grep -Fwf <(cut -f1 out/tcga-covariates.csv)) \
      <(cut -f2- out/tcga-covariates.csv) | tr '\t' ' ' > out/gwas-eur/hg19-qc.tcga-covariates.csv
paste <(cut -f1-11 out/gwas-eur/hg19-nold-qc.eigenvec | grep -Fwf <(cut -f1 out/tcga-covariates.csv)) \
      <(cut -f2- out/tcga-covariates.csv) | tr '\t' ' ' > out/gwas-eur/hg19-nold-qc.tcga-covariates.csv
    # Takes ~15 min

Now, let's run GWAS on the cleaned-up data:

In [ ]:
%%bash

# Run association tests
plink2 --pfile out/gwas-eur/hg19-qc \
       --covar out/gwas-eur/hg19-qc.tcga-covariates.csv  \
       --pheno out/kegg-all-metabolic-phenotypes.txt \
       --glm hide-covar --adjust --pfilter 1 \
       --out out/gwas-eur/hg19-qc
    # Takes ~50min
plink2 --pfile out/gwas-eur/hg19-nold-qc \
       --covar out/gwas-eur/hg19-nold-qc.tcga-covariates.csv  \
       --pheno out/kegg-all-metabolic-phenotypes.txt \
       --glm hide-covar --adjust --pfilter 1 \
       --out out/gwas-eur/hg19-nold-qc
    # Takes ~1h

# Clumping
parallel "awk '(\$12 + 0) < 5.83E-9' {} > {}.top" ::: out/gwas-eur/hg19-*.linear
parallel "awk '(\$10 + 0) < 0.05'    {} > {}.top" ::: out/gwas-eur/hg19-*.adjusted
# Remove files with no significant SNPs
find out/gwas-eur -type f -name "*.top" -exec sh -c 'if [ $(wc -l < "$0") -eq 1 ]; then rm -f "$0"; fi' {} \;
    # 1m
parallel --bar 'plink2 --pfile out/gwas-eur/hg19-qc --clump {} --clump-allow-overlap --clump-kb 250 --clump-p1 0.00000000583 --clump-p2 0.01 --clump-r2 0.2 --clump-snp-field ID --clump-field P --out {}.clump' ::: out/gwas-eur/hg19-qc.*.top
parallel --bar 'plink2 --pfile out/gwas-eur/hg19-nold-qc --clump {} --clump-allow-overlap --clump-kb 250 --clump-p1 0.00000000583 --clump-p2 0.01 --clump-r2 0.2 --clump-snp-field ID --clump-field P --out {}.clump' ::: out/gwas-eur/hg19-nold-qc.*.top

# Final results
parallel "awk NF {} | awk '{ print \$0,\"{= s:out/gwas-eur/hg19.*-qc\.(hsa.+).glm.+:\1: =}\" }' | sed 1d | awk '{print \$3,\$5,\$6,\$(NF)}'" ::: out/gwas-eur/*.clumps
    # 4m

# Clumping

In [ ]:
%%R

# Manhattan QQ plots...

#------------------------------------------------gene-level

data=read.table("All_top_genes_with_suggestive",header=TRUE,fill = TRUE,row.names=NULL)
head(data,3)

data2=read.table("All_top_genes_sample",header=TRUE,fill = TRUE,row.names=NULL)
head(data2,3)

#Gene-level

manhattan(
  data2,
  chr = "CHR",
  bp = "START",
  p = "P",
  snp = "HGNC_Symbol",
  col = c("darkcyan","darkblue", "skyblue","lightblue","lightgray"),
  chrlabs = NULL,
  suggestiveline = -log10(2.9e-05),
  genomewideline = -log10(2.8e-06),
  highlight = NULL,
  logp = TRUE,
  annotatePval = 0.000028,
  annotateTop = FALSE
)


data=read.table("All_top_SNPS_sample",header=TRUE,fill = TRUE,row.names=NULL)
head(data,3)

#SNP-level
library(GWASinspector)
manhattan.plot(
  data,
  chr = "CHROM",
  pvalue= "P",
  position= "POS",
  "All_metabolic_targets_top_SNPs.png",
  plot.title = "SNP-level Manhattan Plot",
  plot.subtitle = "",
  p.threshold = 0.9,
  sig.threshold.log = -log10(5 * 10^-8),
  beta = "ID",
  std.error = NULL,
  check.columns = TRUE
)

#SNP-level
manhattan(
  data,
  chr = "CHROM",
  bp = "POS",
  p = "P",
  snp = "ID",
  col = c("darkcyan","darkblue", "skyblue","lightblue","lightgray"),
  chrlabs = NULL,
  suggestiveline = -log10(1e-06),
  genomewideline = -log10(5e-08),
  highlight = NULL,
  logp = TRUE,
  annotatePval = 0.000028,
  annotateTop = FALSE
)

qq(data$P)
#------------------------------------------------SNP-level

#Example:
data=read.table("all_chr.rsq0.5_EUR_QC_annotated.KEGG_VALINE_LEUCINE_AND_ISOLEUCINE_BIOSYNTHESIS.glm.linear",header=TRUE,fill = TRUE)
head(data,3)

#SNP-level
library(GWASinspector)
manhattan.plot(
  data,
  chr = "CHROM",
  pvalue= "P",
  position= "POS",
  "KEGG_VALINE_LEUCINE_AND_ISOLEUCINE_BIOSYNTHESIS_SNPs.png",
  plot.title = "",
  plot.subtitle = "",
  p.threshold = 0.9,
  sig.threshold.log = -log10(5 * 10^-8),
  beta = "BETA",
  std.error = NULL,
  check.columns = TRUE
)

library(fastman)
library(qqman)

par(pty="s")
qq(data$P, xlim = c(0, 40), ylim = c(0, 40))




# Gene-set analysis

## MAGMA

In [ ]:
%%bash

# SNP annotation using MAGMA
magma --annotate --snp-loc out/all_chr_EUR_QC_annotated.bim --gene-loc data/NCBI37.3.gene.loc --out out/all_chr_EUR_QC_annotated
    # 10s

In [ ]:
%%R
# Make reference file for mapping Entrez IDs and HGNC symbols
mart = biomaRt::useMart(biomart="ENSEMBL_MART_ENSEMBL", dataset="hsapiens_gene_ensembl", host="https://apr2020.archive.ensembl.org")
genes <- getBM(attributes=c("hgnc_symbol", "entrezgene_id"), mart=mart)
write.table(genes, "out/entrez-ids.txt", row.names=FALSE, col.names=TRUE)

In [ ]:
%%bash

mkdir -p out/magma-gene
# parallel --bar "awk '{print \$3,\$11,\$15}' {} > {}.pval" ::: out/plink-association/*.glm.linear
# parallel --bar 'magma --bfile out/all_chr_EUR_QC_annotated --pval {} use=ID,P ncol=OBS_CT --gene-annot out/all_chr_EUR_QC_annotated.genes.annot --out out/magma-gene/{/.}.magma' ::: out/plink-association/*.pval
    # 1h 25m

for f in out/magma-gene/*.genes.out
do
    sort -k9g $f > ${f}.sorted
    awk '{print $1}' ${f}.sorted > ${f}.entrez
    awk 'NR==FNR {A[$1]; next} ($2 in A)' ${f}.sorted out/entrez-ids.txt |\
        awk '{print $2, $1}' |\
        awk 'FNR == NR {lineno[$1] = NR; next} {print lineno[$1], $0}' <(sed 1d ${f}.entrez) - |\
        sort -k1,1g |\
        cut -d' ' -f2- |\
        awk 'NR==1 {print "Entrez_Id\tHGNC_Symbol"} {print}' > ${f}.hgnc
    paste -d'\t' ${f}.sorted ${f}.hgnc > ${f%.genes.out}.annotated
done
    #  20s


## A-LAVA

In [ ]:
%%bash
mkdir -p out/alava
parallel --plus "awk '{print \$8}' {} | sed 1d > out/alava/{/..}" ::: out/magma-gene/*.genes.out
python alava.py out/alava
parallel "paste -d' ' data/65_metabolic_KEGG_Ids_2 {} | sort -k5rn | tee {}.sorted | awk '(\$5 + 0) > 1.64' | awk 'NF>2' > {}.alava-top" ::: out/alava/*.alava
cat out/alava/*.alava-top > out/alava-top.txt

MAGMA (for comparison):

In [ ]:
%%bash

# MAGMA.py???
# parallel 'python ../../MAGMA.py {}' ::: out/alava/*.alava

parallel "paste -d' ' data/65_metabolic_KEGG_Ids_2 <(sort -k1n {}) | tee {}.kegg-annotated | sort -k6rn | awk '(\$6 + 0) > 1.64' | awk 'NF>2' > {}.magma-top" ::: out/alava/*.alava
cat out/alava/*.magma-top > out/magma-top.txt